In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

load_dotenv()

In [ ]:
models = ChatGroq(model='llama-3.3-70b-versatile')

In [ ]:
class CustTick(TypedDict):
    query: str
    intent: str

In [ ]:
def classifier(state: CustTick) -> CustTick:
    query = state["query"]
    prompt = f"""
                You are a customer support ticket classifier.

                Classify the ticket into exactly one category:

                - billing: payments, refunds, invoices, subscriptions, charges
                - technical: bugs, errors, crashes, login issues, system problems
                - general: all other questions or requests

                Ticket:
                {query}
                """
    state["intent"] = models.invoke(prompt).content
    return state

In [ ]:
def billing_node(state: CustTick) -> CustTick:
    print("\n--- [Routing to: Billing Department] ---")
    # You can add specific logic here, e.g., connecting to Stripe API or billing DB
    return state

def technical_node(state: CustTick) -> CustTick:
    print("\n--- [Routing to: Technical Support] ---")
    # You can add specific logic here, e.g., logging a bug report in Jira
    return state

def general_node(state: CustTick) -> CustTick:
    print("\n--- [Routing to: General Support] ---")
    # You can add specific logic here, e.g., replying with a generic template
    return state

In [ ]:
def route_ticket(state: CustTick) -> str:
    # Normalize the intent to handle whitespace, case variation, and surrounding text
    intent = state["intent"].lower().strip()
    
    if "billing" in intent:
        return "billing"
    elif "technical" in intent:
        return "technical"
    else:
        return "general"

In [ ]:
graph = StateGraph(CustTick)

# Add all the nodes to the graph
graph.add_node('classifier', classifier)
graph.add_node('billing', billing_node)
graph.add_node('technical', technical_node)
graph.add_node('general', general_node)

# Add normal start edge
graph.add_edge(START, 'classifier')

# Add conditional routing edge
graph.add_conditional_edges(
    'classifier',
    route_ticket,
    {
        'billing': 'billing',
        'technical': 'technical',
        'general': 'general'
    }
)

# Connect each leaf node to the end
graph.add_edge('billing', END)
graph.add_edge('technical', END)
graph.add_edge('general', END)

workflow = graph.compile()

In [ ]:
# Test the workflow with different intents
test_queries = [
    "I was charged twice for my subscription.",
    "The application crashes every time I click upload.",
    "What are your operating hours during holidays?"
]

for query in test_queries:
    print(f"\nProcessing Query: '{query}'")
    initial_state = {'query': query}
    output = workflow.invoke(initial_state)
    print(f"Detected Intent: {output['intent'].strip()}")